# Tracking Data + Git Workshop

**Triton Ball** · SkillCorner Open Data · kloppy

Today: take one off-ball run from a real A-League match, measure it from raw tracking data, and animate it. Along the way you ship two pull requests.

| Min | Part | You do | Git |
|---|---|---|---|
| 10–20 | 1. Load and explore | run cells | stay on `main` |
| 20–35 | 2. Find off-ball runs | write `get_off_ball_runs` in `src/events.py` | PR #1 `feature/off-ball-runs` |
| 35–50 | 3. Speed from positions | write `add_velocity` in `src/kinematics.py` | PR #2 `feature/kinematics` |
| 50–60 | 4. Plot and animate | run cells, enjoy | — |

Grey cells are worked examples: just run them. A **Check** cell follows each task and prints *All checks passed* when your code works.

Data: [SkillCorner Open Data](https://github.com/SkillCorner/opendata). Credit SkillCorner if you share anything you build.

## Part 0: Setup (run during the slides)

### Are you in your virtual environment?

A **virtual environment** (venv) is a private folder of Python packages for one project. Without one, every project on your laptop shares the same packages, so upgrading kloppy for this workshop could break another class's code. You made yours with `python -m venv .venv` and registered it with Jupyter as **tracking-workshop** (see the README).

The notebook only uses the venv if Jupyter runs it on that **kernel**. Check the kernel name in the top-right corner: it should say **tracking-workshop**. If it doesn't, click it and switch.

In [ ]:
import sys

in_venv = sys.prefix != sys.base_prefix
print("Python running this notebook:", sys.executable)
print("Inside a virtual environment:", in_venv)
if not in_venv:
    print("Not in the venv. Switch the kernel (top right) to 'tracking-workshop', then re-run this cell.")

### Load the data

Run the next three cells now. The last one downloads a full match of tracking data and takes **1–2 minutes**, so it can load while we talk about Git.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from src import load, events, kinematics, viz

pd.set_option("display.max_columns", 40)
DATA_DIR = os.environ.get("WORKSHOP_DATA_DIR")  # None = stream from SkillCorner's GitHub
MATCH_ID = 1925299  # Brisbane Roar vs Perth Glory, 21 Dec 2024

In [ ]:
ev = load.load_events(MATCH_ID, DATA_DIR)
ph = load.load_phases(MATCH_ID, DATA_DIR)
print(len(ev), "events,", len(ph), "phases")

In [ ]:
tracking = load.load_tracking(MATCH_ID, data_dir=DATA_DIR)   # 1-2 minutes
tdf = load.tracking_to_df(tracking)
players = load.player_lookup(tracking)
print(tdf.shape)

## Part 1: Load and explore

Three files per match, glued together by the **frame** number (10 frames per second):

| File | One row per | Key columns |
|---|---|---|
| tracking | frame | x, y of every player and the ball |
| dynamic events | event | event_type, event_subtype, player, frame_start, frame_end |
| phases of play | phase | team in possession, phase type, frame_start, frame_end |

### Tracking

In [ ]:
meta = tracking.metadata
print("Teams:", [t.name for t in meta.teams])
print("Frame rate:", meta.frame_rate, "fps | Pitch:",
      meta.pitch_dimensions.pitch_length, "x", meta.pitch_dimensions.pitch_width, "m")
tdf.iloc[:3, :12]

One row per frame. Every player gets `<player_id>_x` and `<player_id>_y` (NaN when not on the pitch). Coordinates are **meters from the center spot**: x along the length, y along the width.

In [ ]:
players.head(12)

### Dynamic events: what values are available?

In [ ]:
ev["event_type"].value_counts()

In [ ]:
ev.groupby("event_type")["event_subtype"].value_counts()

In [ ]:
# One off-ball run, showing only the columns that are filled in
ev[ev["event_type"] == "off_ball_run"].iloc[0].dropna().head(40)

### Phases of play

In [ ]:
ph[["frame_start", "frame_end", "duration", "team_in_possession_shortname",
    "team_in_possession_phase_type", "team_out_of_possession_phase_type"]].head(8)

In [ ]:
# Each event's phase_index points at a row of the phases table
events.add_phase_info(ev, ph).loc[ev["event_type"] == "off_ball_run",
    ["event_id", "event_subtype", "team_in_possession_phase_type", "phase_frame_start", "phase_frame_end"]].head()

## Part 2: Find the off-ball runs (PR #1)

Make a branch for this part. In a terminal:

```bash
git switch -c feature/off-ball-runs
```

SkillCorner classifies 10 run types: `dropping_off`, `coming_short`, `pulling_wide`, `pulling_half_space`, `support`, `run_ahead_of_the_ball`, `overlap`, `underlap`, `behind`, `cross_receiver`.

In [ ]:
obr = ev[ev["event_type"] == "off_ball_run"]
pd.crosstab(obr["event_subtype"], obr["team_in_possession_phase_type"])

### Your turn: `get_off_ball_runs`

Open **`src/events.py`**, implement `get_off_ball_runs` (the docstring has hints), save, and run the check.

In [ ]:
# Check
r = events.get_off_ball_runs(ev)
assert (r["event_type"] == "off_ball_run").all(), "keep only off_ball_run events"
assert r["frame_start"].is_monotonic_increasing, "sort by frame_start"
assert list(r.index[:3]) == [0, 1, 2], "reset the index"
r = events.get_off_ball_runs(ev, subtype=["overlap", "underlap"], phase="create")
assert set(r["event_subtype"]) <= {"overlap", "underlap"} and set(r["team_in_possession_phase_type"]) <= {"create"}
r = events.get_off_ball_runs(ev, targeted=True, min_duration=2)
assert r["targeted"].all() and (r["duration"] >= 2).all()
print("All checks passed")

In [ ]:
# Runs in behind that a teammate tried to pass to, longest first
candidates = events.get_off_ball_runs(ev, subtype="behind", targeted=True)
candidates[events.RUN_COLUMNS].sort_values("distance_covered", ascending=False).head(8)

In [ ]:
RUN_ID = "1_181" if MATCH_ID == 1925299 else candidates.sort_values("distance_covered").iloc[-1]["event_id"]
run = ev.loc[ev["event_id"] == RUN_ID].iloc[0]
RUNNER, RUNNER_TEAM = int(run["player_id"]), int(run["team_id"])
F0, F1 = int(run["frame_start"]), int(run["frame_end"])
print(f"{run['player_name']} ({run['team_shortname']}): {run['event_subtype']} run in a {run['team_in_possession_phase_type']} phase")
print(f"frames {F0}-{F1}, {run['duration']} s, {run['distance_covered']} m, SkillCorner avg speed {run['speed_avg']} km/h")

### Git checkpoint: ship `feature/off-ball-runs`

In a terminal, from the repo folder:

```bash
git status                     # what changed?
git diff                       # read your own changes
git add src/events.py
git commit -m "Add get_off_ball_runs filter"
git push -u origin feature/off-ball-runs
```

On GitHub: open **your fork** → *Compare & pull request*. Make sure the base is **your fork's** `main` (not the workshop repo) → fill in the template → **Create pull request**. Ask a neighbor for one review comment, then **Merge** and **Delete branch**.

Then bring your laptop up to date:

```bash
git switch main
git pull
```

## Part 3: Speed from positions (PR #2)

```bash
git switch -c feature/kinematics
```

Tracking only gives positions. Speed is **distance / time**: at 10 fps one frame is 0.1 s. Positions jitter a little every frame, so we smooth them first (already written for you).

In [ ]:
PAD = 30  # 3 s of context either side of the run
seq = tdf[(tdf["frame_id"] >= F0 - PAD) & (tdf["frame_id"] <= F1 + PAD)].reset_index(drop=True)
xy = kinematics.smooth_positions(kinematics.player_xy(seq, RUNNER))
xy.head()

### Your turn: `add_velocity`

Open **`src/kinematics.py`** and implement `add_velocity`: `vx`, `vy`, `speed` (m/s) and `speed_kmh`. The check uses a fake player running in a straight line at exactly 5 m/s.

In [ ]:
# Check
fake = pd.DataFrame({"frame_id": range(20), "period_id": 1, "x": np.arange(20) * 0.5, "y": np.zeros(20)})
v = kinematics.add_velocity(fake)
assert np.allclose(v["speed"].iloc[1:], 5), "0.5 m per frame at 10 fps is 5 m/s"
assert np.allclose(v["speed_kmh"].iloc[1:], 18), "5 m/s = 18 km/h"
assert np.allclose(v["vy"].iloc[1:], 0)
assert np.allclose(kinematics.add_velocity(fake.drop(index=[6, 7]))["speed"].iloc[6], 5), "use frame_id for dt"
print("All checks passed")

In [ ]:
# Your add_velocity now powers the full pipeline: acceleration and direction build on it
kin = kinematics.compute_kinematics(seq, RUNNER)
during = kin["frame_id"].between(F0, F1)
print(f"Our avg speed during the run: {kin.loc[during, 'speed_kmh'].mean():.1f} km/h | SkillCorner: {run['speed_avg']:.1f} km/h")
print(f"Top speed: {kin['speed_kmh'].max():.1f} km/h | peak acceleration: {kin['acceleration'].max():.1f} m/s²")

In [ ]:
# Distance to the nearest opponent, every frame
opponents = players.loc[players["team_id"] != RUNNER_TEAM, "player_id"]
nearest = kinematics.nearest_opponent(seq, RUNNER, opponents)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(kin["frame_id"], kin["speed_kmh"], color="#d62728")
ax1.set_ylabel("speed (km/h)", color="#d62728")
ax1.set_xlabel("frame")
ax2 = ax1.twinx()
ax2.plot(nearest["frame_id"], nearest["nearest_opponent_dist"], color="#1f77b4")
ax2.set_ylabel("nearest opponent (m)", color="#1f77b4")
for f in (F0, F1):
    ax1.axvline(f, color="grey", ls="--", lw=1)
ax1.set_title(f"{run['player_name']}: speed vs separation (dashed = run start/end)")
plt.show()

### Git checkpoint: ship `feature/kinematics`

In a terminal, from the repo folder:

```bash
git status                     # what changed?
git diff                       # read your own changes
git add src/kinematics.py
git commit -m "Compute player velocity and speed"
git push -u origin feature/kinematics
```

On GitHub: open **your fork** → *Compare & pull request*. Make sure the base is **your fork's** `main` (not the workshop repo) → fill in the template → **Create pull request**. Ask a neighbor for one review comment, then **Merge** and **Delete branch**.

Then bring your laptop up to date:

```bash
git switch main
git pull
```

## Part 4: Plot and animate

The drawing code in `src/viz.py` is complete. Read it later to see how mplsoccer and `FuncAnimation` work.

In [ ]:
L, W = meta.pitch_dimensions.pitch_length, meta.pitch_dimensions.pitch_width
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
viz.plot_frame(tdf, F0, players, highlight=RUNNER, kinematics=kin, pitch_length=L, pitch_width=W,
               ax=axes[0], title=f"Run starts (frame {F0})")
viz.plot_frame(tdf, F1, players, highlight=RUNNER, kinematics=kin, pitch_length=L, pitch_width=W,
               ax=axes[1], title=f"Run ends (frame {F1})")
plt.show()

### Your velocity, animated

The animation runs `kinematics.compute_kinematics` for **every player**, and that is built on the `add_velocity` you wrote in Part 3:

- each **arrow** is a velocity vector `(vx, vy)`, drawn to where the player will be in 2 seconds at their current velocity
- the runner's **gold arrow** and the label (speed, acceleration, heading) come from his kinematics table, `kin`

Watch Brisbane's arrows swing round toward their own goal as the turnover happens. Takes about 30 seconds to build.

In [ ]:
anim = viz.animate_sequence(tdf, F0 - PAD, F1 + PAD, players, highlight=RUNNER, kinematics=kin,
                            trail=15, pitch_length=L, pitch_width=W,
                            show_velocity=True, arrow_seconds=2)
HTML(anim.to_jshtml())

In [ ]:
from matplotlib import animation
anim.save(ROOT / "run.gif", writer=animation.PillowWriter(fps=10))  # .gif is gitignored
print("saved", ROOT / "run.gif")

## After the workshop

Pick one, each on its own branch with its own PR:

1. Change `RUN_ID` to another run (or `MATCH_ID` to another match) and animate it.
2. Animate the 5 fastest runs (`events.get_off_ball_runs(ev).sort_values("speed_avg")`).
3. Count opponents within 5 m of the runner each frame with `kinematics.distances_from_player`.
4. Compare your average speed with SkillCorner's `speed_avg` for 20 runs. Which smoothing window gets closest?
5. Change `arrow_seconds`, or color each arrow by acceleration instead of team.